# Pose training notebook

Cleaned version of the original training notebook.

The main change is a single `FOREST_ENABLED` switch. When it is `False`, forest data is not loaded, forest batches are not processed, forest metrics are not logged, and forest plots are not created.


In [15]:
%reload_ext autoreload
%autoreload 2

import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from dataset import SceneTwoPairsDataset
from augmentations import build_train_transform, build_eval_transform
from colmap_pair_dataset import ColmapPairDataset
from model import PairImageCylinderModel
from losses import (
    supervised_loss,
    matched_radius_consistency_loss,
    matched_reprojection_loss_2d,
    compute_supervised_pair_losses,
    compute_forest_loss,
    patch_correspondence_loss,
)
from metrics import pose_errors
from debugger import bug_check, debug_sinkhorn_matching
from utils import plot_estimated_cylinders_on_images, plot_relative_pose, pose_to_text


In [16]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

BATCH_SIZE = 16
NUM_EPOCHS = 300
VAL_INTERVAL = 10
LEARNING_RATE = 1e-4

# Checkpoint settings
BEST_MODEL_PATH = "best_model.pt"
LATEST_MODEL_PATH = "latest_model.pt"
# Choose which trained weights to use for test/inference: "best" or "latest".
INFERENCE_MODEL = "latest"

# Forest controls
FOREST_ENABLED = False
FOREST_START_EPOCH = 310

# Loss settings
LAMBDA_OCC = 10.0
LAMBDA_RADIUS = 10.0
LAMBDA_RAD = 10.0
LAMBDA_REPROJ = 1.0
LAMBDA_FOREST_SPARSITY = 0.1
OCC_THRESH = 0.5
LAMBDA_CORR = 0.2

# Visualization settings
INFERENCE_BATCH_SIZE = 4
INFERENCE_SHOW_TEXT = False
INFERENCE_SHOW_IMAGES = True
FOREST_SAMPLE_IDX = 0
FOREST_OCC_THRESHOLD = 0.3
VISION_SAMPLE_IDX = 0


In [17]:
# Device and model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

model = PairImageCylinderModel(
    img_size=128,
    patch_size=16,
    in_chans=3,
    embed_dim=192,
    depth=4,
    num_heads=4,
    num_bins=128,
    dropout=0.1,
).to(device)

optimizer = torch.optim.AdamW( model.parameters(), lr=LEARNING_RATE, weight_decay=0.05, )

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params / 1e6:.2f}M)")


CUDA available: False
Model parameters: 2,806,151 (2.81M)


## Data

Training and validation data are always available. The forest loader is created only when `FOREST_ENABLED=True`.


In [18]:
train_dataset = SceneTwoPairsDataset(
    root_dir="dataset",
    image_size=128,
    debug=False,
    return_two_pairs=True,
    transform=build_train_transform(128),
)
train_loader = DataLoader( train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, )

val_dataset = SceneTwoPairsDataset(
    root_dir="valdataset",
    image_size=128,
    debug=False,
    return_two_pairs=False,
    transform=build_eval_transform(128),
)
val_loader = DataLoader( val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, )

forest_loader = None
if FOREST_ENABLED:
    forest_dataset = ColmapPairDataset( dataset_dir="forestdataset_torpa", image_size=128, )
    forest_loader = DataLoader( forest_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, )

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
if FOREST_ENABLED:
    print("Forest batches:", len(forest_loader))


Train batches: 36
Val batches: 7


## Optional debug checks

Run these only when needed. They do not affect training.


In [19]:
# bug_check(0)
# debug_sinkhorn_matching(device=device)


In [20]:
# -----------------------------------------------------------------------------
# Training helpers
# -----------------------------------------------------------------------------

def validate():
    model.eval()
    totals = {
        "total": 0.0,
        "supervised": 0.0,
        "vision": 0.0,
        "pose": 0.0,
        "radius_consistency": 0.0,
        "reprojection": 0.0,
        "translation_error": 0.0,
        "translation_magnitude_error": 0.0,
        "translation_direction_error": 0.0,
        "yaw_error": 0.0,
        }

    num_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            img_a, vision_a, img_b, vision_b, pose_ab = tuple( x.to(device, non_blocking=True) for x in batch )

            pred_vision_a, pred_vision_b, pred_pose = model(img_a, img_b)

            trans_err, trans_mag_err, trans_dir_err, yaw_err = pose_errors( pred_pose, pose_ab, )

            totals["translation_error"] += trans_err.item()
            totals["translation_magnitude_error"] += trans_mag_err.item()
            totals["translation_direction_error"] += trans_dir_err.item()
            totals["yaw_error"] += yaw_err.item()

            loss_out = supervised_loss(
                pred_vision_a,
                vision_a,
                pred_vision_b,
                vision_b,
                pred_pose,
                pose_ab,
                occ_thresh=OCC_THRESH,
                lambda_occ=LAMBDA_OCC,
                lambda_radius=LAMBDA_RADIUS,
            )

            radius_cons = matched_radius_consistency_loss(
                pred_vision_a,
                vision_a,
                pred_vision_b,
                vision_b,
                relative_pose_pred=pred_pose,
                matching_mode="gt",
                occ_thresh=OCC_THRESH,
            )
            reproj = matched_reprojection_loss_2d(
                pred_vision_a,
                vision_a,
                pred_vision_b,
                vision_b,
                pred_pose,
                matching_mode="gt",
                occ_thresh=OCC_THRESH,
                fov_degrees=90.0,
            )

            sup_loss = loss_out["total"]
            totals["total"] += sup_loss.item()
            totals["supervised"] += sup_loss.item()
            totals["vision"] += loss_out["vision_a"].item()
            totals["pose"] += loss_out["pose"].item()
            totals["radius_consistency"] += radius_cons.item()
            totals["reprojection"] += reproj.item()
            num_batches += 1

    if num_batches == 0:
        raise RuntimeError("Validation loader is empty; cannot select a best model.")

    return {key: value / num_batches for key, value in totals.items()}


## Training

The individual loss components are kept separate so the total loss can be tuned directly in the training loop.

For example:

`loss = sup_loss + 5.0 * radius_cons + 10.0 * reproj + 10.0 * forest_loss`


In [ ]:
history = {
    "epoch": [],
    "total": [],
    "supervised": [],
    "vision": [],
    "pose": [],
    "translation_error": [],
    "correspondence": [],
    "translation_magnitude_error": [],
    "translation_direction_error": [],
    "yaw_error_deg": [],
    "radius_consistency": [],
    "reprojection": [],
    "forest_epoch": [],
    "forest_total": [],
    "forest_radius_consistency": [],
    "forest_reprojection": [],
    "forest_sparsity": [],
    "val_epoch": [],
    "val_total": [],
    "val_supervised": [],
    "val_vision": [],
    "val_pose": [],
    "val_translation_error": [],
    "val_translation_magnitude_error": [],
    "val_translation_direction_error": [],
    "val_yaw_error_deg": [],
    "val_radius_consistency": [],
    "val_reprojection": [],
}

best_val_loss = float("inf")
best_epoch = None

for epoch in range(NUM_EPOCHS):
    epoch_num = epoch + 1
    model.train()

    forest_active = ( FOREST_ENABLED and forest_loader is not None and epoch_num >= FOREST_START_EPOCH )
    forest_iter = iter(forest_loader) if forest_active else None

    train_totals = {
        "total": 0.0,
        "supervised": 0.0,
        "vision": 0.0,
        "pose": 0.0,
        "translation_error": 0.0,
        "correspondence": 0.0,
        "translation_magnitude_error": 0.0,
        "translation_direction_error": 0.0,
        "yaw_error": 0.0,
        "radius_consistency": 0.0,
        "reprojection": 0.0,
    }

    forest_totals = { "total": 0.0, "radius_consistency": 0.0, "reprojection": 0.0, "sparsity": 0.0, }

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)

        (
            img_a1,
            vision_a1,
            img_b1,
            vision_b1,
            pose_ab1,
            img_a2,
            vision_a2,
            img_b2,
            vision_b2,
            pose_ab2,
        ) = tuple(x.to(device, non_blocking=True) for x in batch)

        pred_vision_a1, pred_vision_b1, pred_pose1, corr_a1, corr_b1 = model(
            img_a1, img_b1, return_corr=True
        )

        pred_vision_a2, pred_vision_b2, pred_pose2, corr_a2, corr_b2 = model(
            img_a2, img_b2, return_corr=True
        )

        corr1 = patch_correspondence_loss(corr_a1, vision_a1, corr_b1, vision_b1)
        corr2 = patch_correspondence_loss(corr_a2, vision_a2, corr_b2, vision_b2)
        corr_loss = 0.5 * (corr1 + corr2)

        trans1, trans_mag1, trans_dir1, yaw1 = pose_errors(pred_pose1, pose_ab1)
        trans2, trans_mag2, trans_dir2, yaw2 = pose_errors(pred_pose2, pose_ab2)

        train_totals["translation_error"] += ( (trans1 + trans2) / 2.0 ).item()
        train_totals["translation_magnitude_error"] += ( (trans_mag1 + trans_mag2) / 2.0 ).item()
        train_totals["translation_direction_error"] += ( (trans_dir1 + trans_dir2) / 2.0 ).item()
        train_totals["yaw_error"] += ( (yaw1 + yaw2) / 2.0 ).item()

        loss_out1, loss_out2, radius_cons, reproj = compute_supervised_pair_losses(
            pred_vision_a1,
            vision_a1,
            pred_vision_b1,
            vision_b1,
            pred_pose1,
            pose_ab1,
            pred_vision_a2,
            vision_a2,
            pred_vision_b2,
            vision_b2,
            pred_pose2,
            pose_ab2,
            occ_thresh=OCC_THRESH,
            lambda_occ=LAMBDA_OCC,
            lambda_radius=LAMBDA_RADIUS,
        )

        sup_loss = (loss_out1["total"] + loss_out2["total"]) / 2.0
        forest_loss = torch.tensor(0.0, device=device)

        if forest_active:
            try:
                batch_forest = next(forest_iter)
            except StopIteration:
                forest_iter = iter(forest_loader)
                batch_forest = next(forest_iter)

            ( img_a_forest, img_b_forest, _, ) = tuple(x.to(device, non_blocking=True) for x in batch_forest)

            ( pred_vision_a_forest, pred_vision_b_forest, pred_pose_forest, ) = model(img_a_forest, img_b_forest)

            forest_metrics_batch = compute_forest_loss(
                pred_vision_a_forest,
                pred_vision_b_forest,
                pred_pose_forest,
                occ_thresh=OCC_THRESH,
                lambda_radius=LAMBDA_RAD,
                lambda_reproj=LAMBDA_REPROJ,
                lambda_sparsity=LAMBDA_FOREST_SPARSITY,
            )

            for key, value in forest_metrics_batch.items():
                forest_totals[key] += value.item()

            forest_loss = forest_metrics_batch["total"]

        # Warm up
        alpha = min(1.0, epoch_num / 30.0)

        loss = sup_loss + LAMBDA_CORR * corr_loss #+ alpha * 0.1 * radius_cons + alpha * 0.1 * reproj + 0.5 * forest_loss

        loss.backward()
        optimizer.step()

        train_totals["total"] += loss.item()
        train_totals["supervised"] += sup_loss.item()
        train_totals["vision"] += ( (loss_out1["vision_a"] + loss_out2["vision_a"]) / 2.0 ).item()
        train_totals["pose"] += ( (loss_out1["pose"] + loss_out2["pose"]) / 2.0 ).item()
        train_totals["radius_consistency"] += radius_cons.item()
        train_totals["reprojection"] += reproj.item()
        train_totals["correspondence"] += corr_loss.item()

    train_metrics = { key: value / len(train_loader) for key, value in train_totals.items() }

    history["epoch"].append(epoch_num)
    history["total"].append(train_metrics["total"])
    history["supervised"].append(train_metrics["supervised"])
    history["vision"].append(train_metrics["vision"])
    history["pose"].append(train_metrics["pose"])
    history["translation_error"].append(train_metrics["translation_error"])
    history["correspondence"].append(train_metrics["correspondence"])
    history["translation_magnitude_error"].append( train_metrics["translation_magnitude_error"] )
    history["translation_direction_error"].append( train_metrics["translation_direction_error"] )
    history["yaw_error_deg"].append(train_metrics["yaw_error"])
    history["radius_consistency"].append(train_metrics["radius_consistency"])
    history["reprojection"].append(train_metrics["reprojection"])

    log = (
        f"Epoch {epoch_num}: "
        f"tot={train_metrics['total']:.4f} | "
        f"sup={train_metrics['supervised']:.4f} | "
        f"vis={train_metrics['vision']:.4f} | "
        f"pose={train_metrics['pose']:.4f} | "
        f"trans={train_metrics['translation_error']:.4f} | "
        f"corr={train_metrics['correspondence']:.4f} | "
        f"trans_mag={train_metrics['translation_magnitude_error']:.4f} | "
        f"trans_dir={train_metrics['translation_direction_error']:.2f}deg | "
        f"radius_cons={train_metrics['radius_consistency']:.4f} | "
        f"reproj={train_metrics['reprojection']:.4f}"
    )

    if forest_active:
        forest_metrics = { key: value / len(train_loader) for key, value in forest_totals.items() }
        history["forest_epoch"].append(epoch_num)
        history["forest_total"].append(forest_metrics["total"])
        history["forest_radius_consistency"].append( forest_metrics["radius_consistency"] )
        history["forest_reprojection"].append(forest_metrics["reprojection"])
        history["forest_sparsity"].append(forest_metrics["sparsity"])

        log += (
            f" | forest_tot={forest_metrics['total']:.4f} | "
            f"forest_radius_cons={forest_metrics['radius_consistency']:.4f} | "
            f"forest_reproj={forest_metrics['reprojection']:.4f} | "
            f"forest_sparse={forest_metrics['sparsity']:.4f}"
        )

    if epoch_num == 1 or epoch_num % VAL_INTERVAL == 0:
        val_metrics = validate()
        history["val_epoch"].append(epoch_num)
        history["val_total"].append(val_metrics["total"])
        history["val_supervised"].append(val_metrics["supervised"])
        history["val_vision"].append(val_metrics["vision"])
        history["val_pose"].append(val_metrics["pose"])
        history["val_translation_error"].append(val_metrics["translation_error"])
        history["val_translation_magnitude_error"].append( val_metrics["translation_magnitude_error"] )
        history["val_translation_direction_error"].append( val_metrics["translation_direction_error"] )
        history["val_yaw_error_deg"].append(val_metrics["yaw_error"])
        history["val_radius_consistency"].append( val_metrics["radius_consistency"] )
        history["val_reprojection"].append(val_metrics["reprojection"])

        is_best = val_metrics["total"] < best_val_loss
        if is_best:
            best_val_loss = val_metrics["total"]
            best_epoch = epoch_num
            torch.save(
                {
                    "epoch": epoch_num,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_metrics": val_metrics,
                    "best_val_loss": best_val_loss,
                    "history": history,
                },
                BEST_MODEL_PATH,
            )

        log += (
            f" | val_tot={val_metrics['total']:.4f} | "
            f"val_sup={val_metrics['supervised']:.4f} | "
            f"val_vis={val_metrics['vision']:.4f} | "
            f"val_pose={val_metrics['pose']:.4f} | "
            f"val_trans={val_metrics['translation_error']:.4f} | "
            f"val_trans_mag={val_metrics['translation_magnitude_error']:.4f} | "
            f"val_trans_dir={val_metrics['translation_direction_error']:.2f}deg | "
            f"val_radius_cons={val_metrics['radius_consistency']:.4f} | "
            f"val_reproj={val_metrics['reprojection']:.4f}"
        )

        if is_best:
            log += " | BEST MODEL SAVED"
    # Save the latest model and complete history after every epoch.
    # This makes the plots available again after restarting VS Code.
    torch.save(
        {
            "epoch": epoch_num,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "history": history,
        },
        LATEST_MODEL_PATH,
    )

    print(log)


# Keep a copy of the final epoch for optional test/inference.
latest_model_state_dict = { key: value.detach().cpu().clone() for key, value in model.state_dict().items() }

print(f"Best validation loss: {best_val_loss:.6f} at epoch {best_epoch}")
print(f"Best model saved to: {BEST_MODEL_PATH}")
print(f"Latest model and history saved to: {LATEST_MODEL_PATH}")


Epoch 1: tot=17.0519 | sup=16.6420 | vis=15.6466 | pose=0.9809 | trans=0.9965 | corr=2.0495 | trans_mag=0.9091 | trans_dir=85.81deg | yaw=10.22deg | radius_cons=0.0041 | reproj=0.0010 | val_tot=13.2507 | val_sup=13.2507 | val_vis=12.2842 | val_pose=0.8781 | val_trans=0.9879 | val_trans_mag=0.8793 | val_trans_dir=80.61deg | val_yaw=4.49deg | val_radius_cons=0.0000 | val_reproj=0.0000 | BEST MODEL SAVED
Epoch 2: tot=13.1994 | sup=12.8154 | vis=11.8649 | pose=0.9303 | trans=0.9935 | corr=1.9197 | trans_mag=0.8649 | trans_dir=84.14deg | yaw=5.70deg | radius_cons=0.0000 | reproj=0.0000
Epoch 3: tot=12.5576 | sup=12.1934 | vis=11.2365 | pose=0.9462 | trans=0.9955 | corr=1.8211 | trans_mag=0.8553 | trans_dir=85.41deg | yaw=5.36deg | radius_cons=0.0000 | reproj=0.0000
Epoch 4: tot=12.1906 | sup=11.8319 | vis=10.8688 | pose=0.9703 | trans=0.9980 | corr=1.7934 | trans_mag=0.8663 | trans_dir=86.84deg | yaw=5.73deg | radius_cons=0.0000 | reproj=0.0000
Epoch 5: tot=11.7707 | sup=11.4180 | vis=10.48

## Load saved training state

Load the latest model and training history so the plots can be recreated after restarting VS Code.


In [ ]:
latest_checkpoint_path = Path(LATEST_MODEL_PATH)
best_checkpoint_path = Path(BEST_MODEL_PATH)
checkpoint_path = ( latest_checkpoint_path if latest_checkpoint_path.exists() else best_checkpoint_path )
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    history = checkpoint["history"]
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))
    best_epoch = checkpoint.get("best_epoch", checkpoint.get("epoch"))
    latest_model_state_dict = { key: value.detach().cpu().clone() for key, value in model.state_dict().items() }
    print( f"Loaded saved model and history from epoch {checkpoint['epoch']} " f"({checkpoint_path})" )
else:
    print( f"No saved checkpoint found at {LATEST_MODEL_PATH} or {BEST_MODEL_PATH}." )


In [ ]:
# -----------------------------------------------------------------------------
# Evaluate the trained correspondence head
# -----------------------------------------------------------------------------

import torch.nn.functional as F


def unique_cylinder_bins(vision, occ_thresh=0.5):
    active = torch.where(vision[:, 0] > occ_thresh)[0]
    ids = vision[active, 3].round().long()

    result = {}
    for cid in ids.unique():
        idx = active[ids == cid]
        if idx.numel() == 1:
            result[int(cid.item())] = int(idx.item())

    return result


def bin_to_patch_col(bin_idx, num_bins, grid_size, fov_degrees=90.0):
    fov = np.deg2rad(fov_degrees)
    theta = -0.5 * fov + bin_idx / (num_bins - 1) * fov

    # flip_x=True, same convention as the correspondence loss
    x = 0.5 - 0.5 * np.tan(theta) / np.tan(0.5 * fov)

    return int(np.clip(x * grid_size, 0, grid_size - 1))


def score_corr_pair(corr_a, vision_a, corr_b, vision_b, temperature=0.1):
    B, P, D = corr_a.shape
    grid = int(P ** 0.5)

    feat_a = F.normalize(corr_a.reshape(B, grid, grid, D).mean(dim=1), dim=-1)
    feat_b = F.normalize(corr_b.reshape(B, grid, grid, D).mean(dim=1), dim=-1)

    results = []

    for i in range(B):
        ids_a = unique_cylinder_bins(vision_a[i])
        ids_b = unique_cylinder_bins(vision_b[i])

        for cid in set(ids_a) & set(ids_b):
            col_a = bin_to_patch_col(ids_a[cid], vision_a.shape[1], grid)
            col_b = bin_to_patch_col(ids_b[cid], vision_b.shape[1], grid)

            # A -> B
            prob_ab = ((feat_a[i, col_a] @ feat_b[i].T) / temperature).softmax(dim=-1)
            pred_b = int(prob_ab.argmax().item())

            results.append({
                "hit": pred_b == col_b,
                "error": abs(pred_b - col_b),
                "prob": prob_ab[col_b].item(),
                "uniform": 1.0 / grid,
            })

            # B -> A
            prob_ba = ((feat_b[i, col_b] @ feat_a[i].T) / temperature).softmax(dim=-1)
            pred_a = int(prob_ba.argmax().item())

            results.append({
                "hit": pred_a == col_a,
                "error": abs(pred_a - col_a),
                "prob": prob_ba[col_a].item(),
                "uniform": 1.0 / grid,
            })

    return results


def evaluate_corr_loader(loader, max_batches=5):
    model.eval()

    results = []
    corr_losses = []

    for batch_idx, batch in enumerate(loader):
        if batch_idx >= max_batches:
            break

        if len(batch) == 5:
            pairs = [(batch[0], batch[1], batch[2], batch[3])]
        else:
            pairs = [
                (batch[0], batch[1], batch[2], batch[3]),
                (batch[5], batch[6], batch[7], batch[8]),
            ]

        for img_a, vision_a, img_b, vision_b in pairs:
            img_a = img_a.to(device)
            img_b = img_b.to(device)
            vision_a = vision_a.to(device)
            vision_b = vision_b.to(device)

            with torch.no_grad():
                _, _, _, corr_a, corr_b = model(img_a, img_b, return_corr=True)

                corr_loss = patch_correspondence_loss(
                    corr_a, vision_a, corr_b, vision_b
                )

            corr_losses.append(corr_loss.item())
            results += score_corr_pair(corr_a, vision_a, corr_b, vision_b)

    return results, np.mean(corr_losses)


def print_corr_results(results, corr_loss, name):
    hit = np.mean([r["hit"] for r in results])
    error = np.mean([r["error"] for r in results])
    prob = np.mean([r["prob"] for r in results])
    uniform = np.mean([r["uniform"] for r in results])

    print(f"\n{name}")
    print("-" * 40)
    print(f"Correspondence loss: {corr_loss:.4f}")
    print(f"Matches:             {len(results)}")
    print(f"Top-1 hit rate:      {100 * hit:.1f}%")
    print(f"Mean column error:   {error:.2f}")
    print(f"Correct probability: {prob:.3f}")
    print(f"Uniform baseline:    {uniform:.3f}")
    print(f"Lift:                {prob / uniform:.2f}x")


train_corr_results, train_corr_loss = evaluate_corr_loader(train_loader, max_batches=5)
val_corr_results, val_corr_loss = evaluate_corr_loader(val_loader, max_batches=5)

print_corr_results(train_corr_results, train_corr_loss, "TRAIN")
print_corr_results(val_corr_results, val_corr_loss, "VALIDATION")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history["epoch"], history["correspondence"])
plt.xlabel("Epoch")
plt.ylabel("Correspondence loss")
plt.title("Training correspondence loss")
plt.grid(True, alpha=0.3)
plt.show()

## Training plots

In [ ]:
epochs = np.array(history["epoch"])
val_epochs = np.array(history["val_epoch"])
forest_epochs = np.array(history["forest_epoch"])

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["supervised"], label="train supervised")
plt.plot(epochs, history["reprojection"], label="train reprojection")
plt.plot(epochs, history["total"], label="train total")


if FOREST_ENABLED and len(forest_epochs) > 0:
    plt.plot(forest_epochs, history["forest_total"], "--", label="forest total")

if len(val_epochs) > 0:
    plt.plot(val_epochs, history["val_supervised"], ".--", label="val supervised")
    plt.plot(val_epochs, history["val_reprojection"], ".--", label="val reprojection")
    plt.plot(val_epochs, history["val_total"], ".--", label="val total")

if FOREST_ENABLED:
    plt.axvline(FOREST_START_EPOCH, linestyle=":", alpha=0.5, label="forest start")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["radius_consistency"], label="train radius_consistency")

if FOREST_ENABLED and len(forest_epochs) > 0:
    plt.plot( forest_epochs, history["forest_radius_consistency"], "--", label="forest radius_consistency", )

if len(val_epochs) > 0:
    plt.plot( val_epochs, history["val_radius_consistency"], ".--", label="val radius_consistency", )

if FOREST_ENABLED:
    plt.axvline(FOREST_START_EPOCH, linestyle=":", alpha=0.5, label="forest start")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Radius consistency loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["reprojection"], label="train reprojection")

if FOREST_ENABLED and len(forest_epochs) > 0:
    plt.plot( forest_epochs, history["forest_reprojection"], "--", label="forest reprojection", )

if len(val_epochs) > 0:
    plt.plot( val_epochs, history["val_reprojection"], ".--", label="val reprojection", )

if FOREST_ENABLED:
    plt.axvline(FOREST_START_EPOCH, linestyle=":", alpha=0.5, label="forest start")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Reprojection loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["translation_error"], label="train translation error")
if len(val_epochs) > 0:
    plt.plot(val_epochs, history["val_translation_error"], ".--", label="val translation error")
plt.xlabel("Epoch")
plt.ylabel("Translation error")
plt.title("Translation error")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["translation_magnitude_error"], label="train translation magnitude error")
if len(val_epochs) > 0:
    plt.plot( val_epochs, history["val_translation_magnitude_error"], ".--", label="val translation magnitude error", )
plt.xlabel("Epoch")
plt.ylabel("Translation magnitude error")
plt.title("Translation magnitude error")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["translation_direction_error"], label="train translation direction error")
if len(val_epochs) > 0:
    plt.plot( val_epochs, history["val_translation_direction_error"], ".--", label="val translation direction error", )
plt.xlabel("Epoch")
plt.ylabel("Translation direction error (degrees)")
plt.title("Translation direction error")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["yaw_error_deg"], label="train yaw error")
if len(val_epochs) > 0:
    plt.plot(val_epochs, history["val_yaw_error_deg"], ".--", label="val yaw error")
plt.xlabel("Epoch")
plt.ylabel("Yaw error (degrees)")
plt.title("Yaw error")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Inference on test data

In [ ]:
# Choose the model used for test/inference:
#   "best"   -> checkpoint with lowest validation loss
#   "latest" -> weights from the final training epoch
# This is independent of whether the best checkpoint is loaded later.
INFERENCE_MODEL = INFERENCE_MODEL.lower()
print(f"INFERENCE_MODEL: {INFERENCE_MODEL}")
if INFERENCE_MODEL not in {"best", "latest"}:
    raise ValueError("INFERENCE_MODEL must be either 'best' or 'latest'")

if INFERENCE_MODEL == "best":
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print( f"Using BEST model from epoch {checkpoint['epoch']} " f"with val_total={checkpoint['best_val_loss']:.6f}" )
else:
    model.load_state_dict(latest_model_state_dict)
    print(f"Using LATEST model from epoch {NUM_EPOCHS}")

model.eval()

infer_dataset = SceneTwoPairsDataset( root_dir="testdataset", return_two_pairs=False, )
infer_loader = DataLoader( infer_dataset, batch_size=INFERENCE_BATCH_SIZE, shuffle=True, num_workers=0, )

batch = next(iter(infer_loader))
img_a, vision_a, img_b, vision_b, pose_ab = batch

img_a = img_a.to(device)
vision_a = vision_a.to(device)
img_b = img_b.to(device)
vision_b = vision_b.to(device)
pose_ab = pose_ab.to(device)

with torch.no_grad():
    pred_vision, pred_vision_b, pred_pose = model(img_a, img_b)

if INFERENCE_SHOW_TEXT:
    n_show = min(4, img_a.shape[0])
    for i in range(n_show):
        print(f"Sample {i}")
        print("  GT   pose:", pose_to_text(pose_ab[i].detach().cpu()))
        print("  Pred pose:", pose_to_text(pred_pose[i].detach().cpu()))
        pose_err = torch.abs(pred_pose[i] - pose_ab[i]).detach().cpu()
        print( f"  |err|: tx={pose_err[0]:.3f}, " f"ty={pose_err[1]:.3f}, " f"sin={pose_err[2]:.3f}, " f"cos={pose_err[3]:.3f}" )
        print()

print("pred_vision shape:", pred_vision.shape)
print("pred_pose shape:", pred_pose.shape)
print("GT pose:", pose_ab[VISION_SAMPLE_IDX].detach().cpu())
print("Pred pose:", pred_pose[VISION_SAMPLE_IDX].detach().cpu())


## Vision prediction plots

In [ ]:
sample_idx = VISION_SAMPLE_IDX
bins = np.arange(pred_vision.shape[1])
occ_threshold = OCC_THRESH

if INFERENCE_SHOW_IMAGES:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    names = ["occupancy", "radius", "depth"]

    gt_occ = vision_a[sample_idx, :, 0].detach().cpu().numpy()
    pr_occ = pred_vision[sample_idx, :, 0].detach().cpu().numpy()

    for j, ax in enumerate(axes):
        gt = vision_a[sample_idx, :, j].detach().cpu().numpy()
        pr = pred_vision[sample_idx, :, j].detach().cpu().numpy()

        ax.plot(bins, gt, label="GT", color="tab:blue")

        if j == 0:
            ax.plot(bins, pr, label="Pred", color="tab:orange")
        else:
            pr_mask = pr_occ > occ_threshold
            ax.scatter( bins[pr_mask], pr[pr_mask], label="Pred", color="tab:orange", s=25, )

        ax.set_ylabel(names[j])
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[-1].set_xlabel("Bin")
    fig.suptitle(f"Vision prediction sample {sample_idx}")
    plt.tight_layout()
    plt.show()

    fig_cyl, axes_cyl = plot_estimated_cylinders_on_images(
        [img_a[sample_idx], img_b[sample_idx]],
        [pred_vision[sample_idx], pred_vision_b[sample_idx]],
        titles=["Image A: pred cylinders", "Image B: pred cylinders"],
        occ_threshold=occ_threshold,
        flip_x=True,
    )
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_a[sample_idx].cpu().permute(1, 2, 0))
    axes[0].set_title("Image A")
    axes[0].axis("off")
    axes[1].imshow(img_b[sample_idx].cpu().permute(1, 2, 0))
    axes[1].set_title("Image B")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    plot_relative_pose(pose_ab[sample_idx], pred_pose[sample_idx])


## Forest visualization

In [ ]:
if FOREST_ENABLED:
    forest_sample_idx = FOREST_SAMPLE_IDX
    model.eval()
    forest_batch = next(iter(forest_loader))
    img_a_forest, img_b_forest, _ = forest_batch

    img_a_forest = img_a_forest.to(device)
    img_b_forest = img_b_forest.to(device)

    with torch.no_grad():
        pred_vision_a_forest, pred_vision_b_forest, pred_pose_forest = model( img_a_forest, img_b_forest, )

    fig_forest_cyl, axes_forest_cyl = plot_estimated_cylinders_on_images(
        [
            img_a_forest[forest_sample_idx],
            img_b_forest[forest_sample_idx],
        ],
        [
            pred_vision_a_forest[forest_sample_idx],
            pred_vision_b_forest[forest_sample_idx],
        ],
        titles=["Forest A: pred cylinders", "Forest B: pred cylinders"],
        occ_threshold=FOREST_OCC_THRESHOLD,
        flip_x=True,
    )
    plt.show()

    print("forest pred_vision_a shape:", pred_vision_a_forest.shape)
    print("forest pred_pose shape:", pred_pose_forest.shape)
    print( "forest pred_pose:", pred_pose_forest[forest_sample_idx].detach().cpu(), )


## Notes

- `FOREST_ENABLED=False` means no forest data is loaded and no forest log/plot is produced.
- `FOREST_ENABLED=True` enables the forest metrics from `FOREST_START_EPOCH` onward.
- `USE_FOREST_IN_TOTAL_LOSS=False` preserves the original optimization behavior.
- Set `USE_FOREST_IN_TOTAL_LOSS=True` when you want the forest objective to affect gradients.
- Validation runs on epoch 1 and every `VAL_INTERVAL` epochs. The checkpoint with the lowest validation `total` loss is saved to `BEST_MODEL_PATH`.
- Test/inference uses the best validation checkpoint when `LOAD_BEST_MODEL_FOR_INFERENCE=True`.
